# Basic DiD

A policy or program reaches some units and not others, and you have outcomes for both
groups before and after it started. How much did the program change the outcome for the
units that received it? This tutorial answers that question with the simplest design in
the library, the 2x2 difference-in-differences (two groups, two periods), and uses it to
introduce the `diff-diff` API.

**What you'll learn**

- What a 2x2 DiD estimates and what has to be true for it to be causal
- How to fit it with column names or with an R-style formula, and how to read the results
- How to add covariates and fixed effects, and when to switch to `TwoWayFixedEffects`
- Why panel data needs clustered standard errors, and how to get them
- How to use the wild cluster bootstrap when there are few clusters

**Prerequisites:** none. This is the starting point for the other tutorials.

In [1]:
import numpy as np
import pandas as pd

from diff_diff import DifferenceInDifferences, TwoWayFixedEffects, generate_did_data

## The estimand and the assumptions

The target is the **average treatment effect on the treated (ATT)**: the average
difference, in the post-treatment period, between what happened to the treated units and
what would have happened to them without treatment. The second quantity is never observed,
so DiD replaces it with an assumption:

- **Parallel trends.** Without treatment, the average outcome of the treated group would
  have changed by the same amount as the average outcome of the comparison group. The
  groups may differ in level; they must not differ in trend.
- **No anticipation.** Treated units do not change their behavior before the treatment
  starts, so the pre-period is a clean baseline.

Under these two assumptions,

$$\text{ATT} = (\bar{Y}_{\text{treated, post}} - \bar{Y}_{\text{treated, pre}}) -
(\bar{Y}_{\text{control, post}} - \bar{Y}_{\text{control, pre}}).$$

What would break it: a shock that hits only one group at the same time as the treatment,
units selecting into treatment because their outcomes were already trending differently,
or a treatment that was announced early enough for units to respond in the pre-period.
Neither assumption can be verified from the data alone.
[Parallel Trends Diagnostics](04_parallel_trends.ipynb) and
[Honest DiD Sensitivity](05_honest_did.ipynb) show how to probe them.

## Simulated data

`generate_did_data` builds a balanced panel with a known treatment effect, so we can check
every estimate against the truth. Parallel trends holds by construction.

In [2]:
TRUE_ATT = 5.0

data = generate_did_data(
    n_units=100,
    n_periods=2,
    treatment_effect=TRUE_ATT,
    treatment_fraction=0.5,
    treatment_period=1,  # periods are 0 (pre) and 1 (post)
    noise_sd=1.0,
    seed=42,
)

print(f"Dataset shape: {data.shape}")
data.head(6)

Dataset shape: (200, 6)


,unit,period,treated,post,outcome,true_effect
0,0,0,1,0,10.231272,0.0
1,0,1,1,1,17.408662,5.0
2,1,0,1,0,7.563768,0.0
3,1,1,1,1,14.157547,5.0
4,2,0,1,0,10.567285,0.0
5,2,1,1,1,16.795465,5.0


In [3]:
# One row per unit and period: 50 treated and 50 control units, each observed twice
print(data.groupby(["treated", "post"]).size().unstack())

post      0   1
treated        
0        50  50
1        50  50


## A first estimate

`DifferenceInDifferences` has an sklearn-like interface: configure the estimator, then call
`fit()` with the column names. `treatment` is the 0/1 group indicator and `post` is the 0/1
post-period indicator.

Each unit appears in the data twice, and its two observations are not independent, so we
cluster the standard errors at the unit with `cluster="unit"`. The inference section
further down explains why this matters.

In [4]:
did = DifferenceInDifferences(cluster="unit")
results = did.fit(data, outcome="outcome", treatment="treated", post="post")

print(results.summary())

             Difference-in-Differences Estimation Results             

Observations:                    200
Treated:                         100
Control:                         100
R-squared:                    0.6453
Variance:                        CR1 cluster-robust at unit, G=100

----------------------------------------------------------------------
Parameter           Estimate    Std. Err.     t-stat      P>|t|      
----------------------------------------------------------------------
ATT                   5.0318       0.2821     17.840     0.0000   ***
----------------------------------------------------------------------

95% Confidence Interval: [4.4721, 5.5915]
CV (SE/abs(ATT)):             0.0561

Signif. codes: '***' 0.001, '**' 0.01, '*' 0.05, '.' 0.1


### Reading the results

- **ATT**: the estimated effect of the treatment on the treated units
- **Std. Err.**: the standard error of that estimate. The `Variance` line above the table
  says how it was computed: cluster-robust at the unit, with 100 clusters
- **t-stat** and **P>|t|**: the test of the null hypothesis that the ATT is zero
- **95% Confidence Interval**: the range of ATT values the data does not reject

Every number in the summary is also an attribute of the results object. Because the data
is simulated, we can compare the estimate with the truth.

In [5]:
lower, upper = results.conf_int

print(f"Estimated ATT: {results.att:.4f}")
print(f"True ATT:      {TRUE_ATT:.4f}")
print(f"Std. error:    {results.se:.4f}")
print(f"95% CI:        [{lower:.4f}, {upper:.4f}]")
print(f"P-value:       {results.p_value:.4g}")
print(f"CI covers the true ATT: {lower <= TRUE_ATT <= upper}")

Estimated ATT: 5.0318
True ATT:      5.0000
Std. error:    0.2821
95% CI:        [4.4721, 5.5915]
P-value:       1.083e-32
CI covers the true ATT: True


## The formula interface

If you prefer R-style formulas, pass `formula=` instead of the column names. The
coefficient on the interaction `treated:post` is the ATT.

In [6]:
results_formula = DifferenceInDifferences(cluster="unit").fit(
    data, formula="outcome ~ treated * post"
)

print(f"Column-name ATT: {results.att:.6f}")
print(f"Formula ATT:     {results_formula.att:.6f}")
print(f"Same estimate:   {np.isclose(results.att, results_formula.att)}")

Column-name ATT: 5.031807
Formula ATT:     5.031807
Same estimate:   True


## Adding covariates

`covariates=` adds control variables to the regression. They help in two situations: when
a covariate predicts the outcome, controlling for it can make the estimate more precise;
and when the groups would only have followed parallel trends among units with similar
characteristics, the covariates are part of the identification argument. The second case
is better handled by the doubly robust estimators in
[Staggered DiD](02_staggered_did.ipynb).

Here we add two covariates that are unrelated to the outcome by construction, to show the
mechanics. The estimate and its standard error should barely move.

In [7]:
rng = np.random.default_rng(2024)  # a different seed from the data, so the draws are independent
unit_size = pd.Series(rng.normal(100, 20, data["unit"].nunique()), index=data["unit"].unique())
data["size"] = data["unit"].map(unit_size)       # fixed over time
data["age"] = rng.normal(10, 3, len(data))       # varies over time

results_cov = DifferenceInDifferences(cluster="unit").fit(
    data,
    outcome="outcome",
    treatment="treated",
    post="post",
    covariates=["size", "age"],
)

print(f"Without covariates: ATT = {results.att:.4f} (SE {results.se:.4f})")
print(f"With covariates:    ATT = {results_cov.att:.4f} (SE {results_cov.se:.4f})")

Without covariates: ATT = 5.0318 (SE 0.2821)
With covariates:    ATT = 5.0143 (SE 0.2798)


In [8]:
# Every coefficient of the underlying regression is available by name
for name, coef in results_cov.coefficients.items():
    print(f"  {name:<14} {coef:>9.4f}")

  const             9.2217
  treated           0.6356
  post              0.4377
  treated:post      5.0143
  size              0.0003
  age               0.0325


## Fixed effects

Fixed effects control for unobserved differences that are constant within a group, for
example a state-level difference in outcome levels. `DifferenceInDifferences` supports two
equivalent ways to include them:

1. `fixed_effects=[...]` adds one indicator variable per level. Simple, and fine for a
   handful of levels.
2. `absorb=[...]` removes the group means before the regression (the within
   transformation). It gives the same estimate without building the indicators, which
   matters when there are thousands of levels.

We need a panel with a grouping variable to show this: 50 units in 5 states, observed for
4 periods, treated from period 2 onward.

In [9]:
TRUE_ATT_PANEL = 4.0

rng = np.random.default_rng(7)
rows = []
for unit in range(50):
    treated = int(unit < 25)
    state = unit % 5  # treated and control units in every state
    unit_effect = rng.normal(0, 2)
    for period in range(4):
        post = int(period >= 2)
        outcome = (
            10.0
            + unit_effect
            + 0.5 * period
            + 1.5 * state
            + TRUE_ATT_PANEL * treated * post
            + rng.normal(0, 0.5)
        )
        rows.append(
            {
                "unit": unit,
                "state": f"state_{state}",
                "period": period,
                "treated": treated,
                "post": post,
                "outcome": outcome,
            }
        )

panel_df = pd.DataFrame(rows)
print(f"Panel: {panel_df.shape[0]} observations, {panel_df['unit'].nunique()} units")
panel_df.head()

Panel: 200 observations, 50 units


,unit,state,period,treated,post,outcome
0,0,state_0,0,1,0,10.151833
1,0,state_0,1,1,0,10.365391
2,0,state_0,2,1,1,14.557164
3,0,state_0,3,1,1,15.275125
4,1,state_1,0,1,0,9.546779


In [10]:
fit_kwargs = dict(outcome="outcome", treatment="treated", post="post")

# State fixed effects as indicator variables
results_fe = DifferenceInDifferences(cluster="unit").fit(
    panel_df, fixed_effects=["state"], **fit_kwargs
)

# The same fixed effects, absorbed by the within transformation
results_absorb = DifferenceInDifferences(cluster="unit").fit(
    panel_df, absorb=["state"], **fit_kwargs
)

print(f"fixed_effects=['state']: ATT = {results_fe.att:.4f} (SE {results_fe.se:.4f})")
print(f"absorb=['state']:        ATT = {results_absorb.att:.4f} (SE {results_absorb.se:.4f})")

fixed_effects=['state']: ATT = 4.0888 (SE 0.1365)
absorb=['state']:        ATT = 4.0888 (SE 0.1365)


## Two-way fixed effects

The natural fixed effect in a panel is the unit itself. `DifferenceInDifferences` is not
the tool for that: the `treated` indicator is constant within a unit, so absorbing unit
effects would make it collinear with them. `TwoWayFixedEffects` is built for this case. It
absorbs unit fixed effects and the pre/post period effect, and the only regressor left is
`treated x post`. It also clusters at the unit by default, so no `cluster=` is needed.

In [11]:
results_twfe = TwoWayFixedEffects().fit(panel_df, unit="unit", **fit_kwargs)
print(results_twfe.summary())

             Difference-in-Differences Estimation Results             

Observations:                    200
Treated:                          25
Control:                          25
R-squared:                    0.8255
Variance:                         CR1 cluster-robust at unit, G=50

----------------------------------------------------------------------
Parameter           Estimate    Std. Err.     t-stat      P>|t|      
----------------------------------------------------------------------
ATT                   4.0888       0.1348     30.341     0.0000   ***
----------------------------------------------------------------------

95% Confidence Interval: [3.8180, 4.3596]
CV (SE/abs(ATT)):             0.0330

Signif. codes: '***' 0.001, '**' 0.01, '*' 0.05, '.' 0.1


On a balanced panel like this one, all of these specifications give the same point
estimate. The fixed effects are constant within the groups being compared, so they cancel
in the difference-in-differences. What they change is the standard error.

In [12]:
results_plain = DifferenceInDifferences(cluster="unit").fit(panel_df, **fit_kwargs)

comparison = pd.DataFrame(
    {
        "ATT": [r.att for r in (results_plain, results_fe, results_absorb, results_twfe)],
        "SE": [r.se for r in (results_plain, results_fe, results_absorb, results_twfe)],
    },
    index=["no fixed effects", "state indicators", "state absorbed", "two-way fixed effects"],
)
print(comparison.round(4))
print(f"\nTrue ATT: {TRUE_ATT_PANEL}")
lower, upper = results_twfe.conf_int
print(f"TWFE 95% CI [{lower:.3f}, {upper:.3f}] covers the true ATT: {lower <= TRUE_ATT_PANEL <= upper}")

                          ATT      SE
no fixed effects       4.0888  0.1351
state indicators       4.0888  0.1365
state absorbed         4.0888  0.1365
two-way fixed effects  4.0888  0.1348

True ATT: 4.0
TWFE 95% CI [3.818, 4.360] covers the true ATT: True


For effects period by period, with calendar-time fixed effects, `TwoWayFixedEffects` has an
event-study mode: `fit(..., unit=, time=, event_study=True, post_periods=)`. It is used in
[Parallel Trends Diagnostics](04_parallel_trends.ipynb) and
[Honest DiD Sensitivity](05_honest_did.ipynb).

## Inference: why and how to cluster

A standard error is only as good as its assumption about which observations are
independent. In DiD data they rarely all are: the same unit is observed repeatedly, and
treatment is usually assigned to whole groups (states, stores, schools) rather than to
individual observations. The rule is to **cluster at the level where treatment is
assigned**, which in a unit-level panel means the unit.

By default `DifferenceInDifferences` computes heteroskedasticity-robust (HC1) standard
errors that treat every row as independent, so on panel or grouped data you pass
`cluster=` yourself, as we did above. Estimators designed for panels, such as
`TwoWayFixedEffects`, cluster at the unit automatically.

To see what is at stake, we simulate data where treatment is assigned to 20 clusters, each
with 10 members observed before and after, and where members of a cluster share a common
component.

In [13]:
TRUE_ATT_CLUSTERED = 2.5

rng = np.random.default_rng(11)
rows = []
for cluster in range(20):
    treated = int(cluster < 10)
    cluster_effect = rng.normal(0, 2)  # shared by every member, in both periods
    for member in range(10):
        for period in (0, 1):
            outcome = (
                10.0
                + cluster_effect
                + 3.0 * period
                + TRUE_ATT_CLUSTERED * treated * period
                + rng.normal(0, 0.5)
            )
            rows.append(
                {
                    "cluster": cluster,
                    "member": member,
                    "treated": treated,
                    "post": period,
                    "outcome": outcome,
                }
            )

clustered_df = pd.DataFrame(rows)
print(f"{clustered_df.shape[0]} observations in {clustered_df['cluster'].nunique()} clusters")

400 observations in 20 clusters


In [14]:
results_hc1 = DifferenceInDifferences().fit(clustered_df, **fit_kwargs)
results_cluster = DifferenceInDifferences(cluster="cluster").fit(clustered_df, **fit_kwargs)

print(f"ATT (identical in both fits): {results_cluster.att:.4f}   (true ATT: {TRUE_ATT_CLUSTERED})")
print(f"HC1 SE, rows treated as independent: {results_hc1.se:.4f}")
print(f"Cluster-robust SE:                   {results_cluster.se:.4f}")
print(f"Ratio, cluster-robust / HC1:         {results_cluster.se / results_hc1.se:.2f}")

ATT (identical in both fits): 2.4537   (true ATT: 2.5)
HC1 SE, rows treated as independent: 0.4416
Cluster-robust SE:                   0.0639
Ratio, cluster-robust / HC1:         0.14


The two standard errors are far apart, and here the cluster-robust one is the *smaller*
of the two. The reason is specific to this design: the cluster component is the same
before and after, so it cancels out of the difference-in-differences estimate, but it is
still in the regression residuals. The HC1 formula counts that variation as if it were
estimation noise. The cluster-robust formula recognizes that it is common to the cluster.

Do not read this as "clustering shrinks standard errors". When the shocks that hit a
cluster persist over time instead of cancelling, ignoring the clustering makes standard
errors too *small*, and the resulting over-rejection is the classic problem documented by
Bertrand, Duflo and Mullainathan (2004). The direction depends on the data; the fix is the
same either way.

One more detail: with clustered standard errors, p-values and confidence intervals use a
t distribution with (number of clusters - 1) degrees of freedom, 19 here, rather than the
much larger number implied by the row count.

### Wild cluster bootstrap for few clusters

Cluster-robust standard errors rely on having many clusters. With few of them (a common
rule of thumb is fewer than 50), the wild cluster bootstrap is more reliable.

`diff-diff` runs the Wild Cluster Restricted bootstrap (Cameron, Gelbach and Miller 2008),
with the same defaults as R's `fwildclusterboot::boottest`. It imposes the null hypothesis,
studentizes with the cluster-robust standard error, and builds the confidence interval by
inverting the bootstrap test, so the reported p-value and interval always agree (zero lies
outside the interval exactly when the p-value is below the significance level). The
bootstrap replaces the p-value and the interval; it does not produce a different standard
error.

In [15]:
results_bootstrap = DifferenceInDifferences(
    cluster="cluster",
    inference="wild_bootstrap",
    n_bootstrap=999,
    bootstrap_weights="rademacher",
    seed=42,
).fit(clustered_df, **fit_kwargs)

print(results_bootstrap.summary())

             Difference-in-Differences Estimation Results             

Observations:                    400
Treated:                         200
Control:                         200
R-squared:                    0.5211
Inference method:         wild_bootstrap
Bootstrap replications:          999
Number of clusters:               20
Test type:                two-tailed

----------------------------------------------------------------------
Parameter           Estimate    Std. Err.     t-stat      P>|t|      
----------------------------------------------------------------------
ATT                   2.4537       0.0639     38.382     0.0010    **
----------------------------------------------------------------------

95% Confidence Interval: [2.3132, 2.5885]
CV (SE/abs(ATT)):             0.0261

Signif. codes: '***' 0.001, '**' 0.01, '*' 0.05, '.' 0.1


In [16]:
inference = pd.DataFrame(
    {
        "p-value": [results_cluster.p_value, results_bootstrap.p_value],
        "CI lower": [results_cluster.conf_int[0], results_bootstrap.conf_int[0]],
        "CI upper": [results_cluster.conf_int[1], results_bootstrap.conf_int[1]],
    },
    index=["cluster-robust, t(19)", "wild cluster bootstrap"],
)
print(inference.round(4))

                        p-value  CI lower  CI upper
cluster-robust, t(19)     0.000    2.3199    2.5875
wild cluster bootstrap    0.001    2.3132    2.5885


The bootstrap p-value is computed from the bootstrap draws and is floored at
1/(draws + 1), so with 999 draws it cannot be smaller than 0.001: the value shown is that
floor, not an exact probability. The two
intervals are close because 20 clusters with a strong effect is an easy case; the bootstrap
matters most when clusters are fewer or unbalanced.

Two options worth knowing: with Rademacher weights and few clusters, once `n_bootstrap`
reaches the number of possible sign patterns (`2**n_clusters <= n_bootstrap` and at most
20 clusters) every pattern is enumerated and the result no longer depends on `seed`; and
`p_val_type="equal-tailed"` gives an equal-tailed interval instead of the default.

## Exporting results

Results objects convert to a dictionary or a one-row DataFrame, which is convenient for
collecting several estimates into one table.

In [17]:
for key, value in results.to_dict().items():
    shown = f"{value:.4f}" if isinstance(value, float) else value
    print(f"  {key}: {shown}")

  att: 5.0318
  se: 0.2821
  t_stat: 17.8397
  p_value: 0.0000
  conf_int_lower: 4.4721
  conf_int_upper: 5.5915
  n_obs: 200
  n_treated: 100
  n_control: 100
  r_squared: 0.6453
  inference_method: analytical
  vcov_type: hc1
  cluster_name: unit
  n_clusters: 100
  df_convention: cluster
  inference_df: 99.0000


In [18]:
results.to_dataframe()

,att,se,t_stat,p_value,conf_int_lower,conf_int_upper,n_obs,n_treated,n_control,r_squared,inference_method,vcov_type,cluster_name,n_clusters,df_convention,inference_df
0,5.031807,0.282056,17.839734,1.082727e-32,4.472146,5.591467,200,100,100,0.645328,analytical,hc1,unit,100,cluster,99.0


## Summary

- A 2x2 DiD estimates the ATT under parallel trends and no anticipation. Neither can be
  verified from the data, so state them and probe them.
- `DifferenceInDifferences` takes column names or a formula and returns the estimate with
  its standard error, test and confidence interval.
- Covariates and fixed effects go in through `covariates=`, `fixed_effects=` and
  `absorb=`; for unit fixed effects use `TwoWayFixedEffects`.
- Cluster standard errors at the level where treatment is assigned. With few clusters,
  use the wild cluster bootstrap.

## When to use it and when not to

Use the 2x2 design when treatment starts at one date for all treated units and you have a
comparison group that was never treated.

Do not use it, or `TwoWayFixedEffects`, when units are treated at different dates: with
staggered adoption a single two-way fixed effects coefficient can be badly biased. Use the
estimators in [Staggered DiD](02_staggered_did.ipynb) instead. With only a handful of
treated units, see [Synthetic DiD](03_synthetic_did.ipynb).

## Next steps

- [Staggered DiD](02_staggered_did.ipynb): units treated at different times
- [Synthetic DiD](03_synthetic_did.ipynb): few treated units, many comparison units
- [Parallel Trends Diagnostics](04_parallel_trends.ipynb): pre-treatment evidence and
  placebo tests
- [Honest DiD Sensitivity](05_honest_did.ipynb): how much the conclusion depends on
  parallel trends

## References

- Bertrand, M., Duflo, E., & Mullainathan, S. (2004). "How Much Should We Trust
  Differences-in-Differences Estimates?" *The Quarterly Journal of Economics*, 119(1),
  249-275.
- Cameron, A. C., Gelbach, J. B., & Miller, D. L. (2008). "Bootstrap-Based Improvements
  for Inference with Clustered Errors." *The Review of Economics and Statistics*, 90(3),
  414-427.